# Imports

In [1]:
# import muon
import numpy as np
import mudata as md
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import davies_bouldin_score
from sklearn.neighbors import NearestNeighbors

# Read Data

Working off of the preprocessed tonsil h5mu

In [2]:
mdata = md.read_h5mu("../../data/tonsil/tonsil_embedded.h5mu")
rna_adata = mdata.mod['RNA']
prot_adata = mdata.mod['Protein']

/opt/anaconda3/lib/python3.13/site-packages/mudata/_core/mudata.py:1598: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/opt/anaconda3/lib/python3.13/site-packages/mudata/_core/mudata.py:1461: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)


In [3]:
rna_moran = sc.metrics.morans_i(rna_adata, obsm="X_spatial_totalVI")

In [4]:
pd.DataFrame(data = {'Moran_I': rna_moran}).to_csv('../results/Moran_I_our_model_metrics.csv', index=False)

# cLISI

In [5]:
def compute_lisi(emb, labels, perplexity=30):
    n_neighbors = int(3 * perplexity)
    nn = NearestNeighbors(n_neighbors=n_neighbors + 1).fit(emb)
    distances, indices = nn.kneighbors(emb)
    indices = indices[:, 1:]

    labels = np.array(labels)
    out = []

    for i in range(emb.shape[0]):
        neigh = labels[indices[i]]
        p = pd.value_counts(neigh) / len(neigh)
        entropy = -(p * np.log(p)).sum()
        out.append(np.exp(entropy))

    return np.array(out)

In [6]:
rna_adata.obs["cLISI_kmeans5"] = compute_lisi(rna_adata.obsm["X_spatial_totalVI"], rna_adata.obs["kmeans5"])
rna_adata.obs["cLISI_louvain"] = compute_lisi(rna_adata.obsm["X_spatial_totalVI"], rna_adata.obs["louvain_eval"])
rna_adata.obs["cLISI_leiden"] = compute_lisi(rna_adata.obsm["X_spatial_totalVI"], rna_adata.obs["leiden_eval"])

/var/folders/td/snqs86gn12g5nxg94yl2nhrr0000gn/T/ipykernel_44491/3353607529.py:12: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  p = pd.value_counts(neigh) / len(neigh)
/var/folders/td/snqs86gn12g5nxg94yl2nhrr0000gn/T/ipykernel_44491/3353607529.py:12: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  p = pd.value_counts(neigh) / len(neigh)
/var/folders/td/snqs86gn12g5nxg94yl2nhrr0000gn/T/ipykernel_44491/3353607529.py:12: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  p = pd.value_counts(neigh) / len(neigh)
/var/folders/td/snqs86gn12g5nxg94yl2nhrr0000gn/T/ipykernel_44491/3353607529.py:12: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  p = pd.v

In [7]:
rna_adata.obs[['cLISI_kmeans5', 'cLISI_louvain', 'cLISI_leiden']].to_csv("../results/LISI_metrics_our_model_totalVI.csv")

# DBI

In [8]:
# Extract embedding and labels
X = rna_adata.obsm["X_spatial_totalVI"]
labels = rna_adata.obs["leiden_eval"].astype(int).values

In [9]:
dbi_list = []

# compute DBI for each single PC (1D DBI)
for i in range(X.shape[1]):
    Xi = X[:, i].reshape(-1, 1)
    dbi_list.append(davies_bouldin_score(Xi, labels))

dbi_list = np.array(dbi_list)
print(dbi_list)

[  9.98594203   5.46304491  32.74540405  85.24903464  15.76995218
   7.62810613   6.77374564   6.33870494   5.47537311   5.05095382
   7.03428768  88.53294763  21.91303639  34.6583758   45.4449621
   6.22144358 119.33211584   9.96846174   6.81594751   4.20211145]


In [10]:
pd.DataFrame(data = {"DBI_leiden": dbi_list}).to_csv("../results/DBI_our_model_metrics.csv", index = False)